In [1]:
import crypten
import torch

# 初始化 Crypten
crypten.init()

In [2]:
class ApproxMaxPool2D(crypten.nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0, dilation=1, ceil_mode=False, device="cpu"):
        super(ApproxMaxPool2D, self).__init__()
        self.kernel_size = kernel_size
        self.stride = stride 
        self.padding = padding
        self.dilation = dilation
        self.ceil_mode = ceil_mode
        self.device = device
        
    def approxmax_pool2d(self, x_enc, kernel_size, stride=None, padding=0, dilation=1, ceil_mode=False, is_print=False):
        if ceil_mode:
            raise NotImplementedError(
                "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
            )
        
        x2_enc = x_enc.square()
    
        x_enc_avg2 = x_enc.mean().square()
        x_enc_var = x_enc.var()
        approx_r = 7 * x_enc_avg2 + 9 * x_enc_var 
    
#   260  26  2.6  0.26  0.026 
#     0.1  10  1000  100000  10000000
        tau, i = 260, 0
        bound = 10             
        r = torch.tensor([0.1],device=self.device) 
        while(1):
            if approx_r.le(r).get_plain_text():
                break
            r *= 100
            tau /= 10
            i += 1
            if i >= bound:
                break
        x2_enc = tau**2 * x2_enc + 1
        
        # 套对应公式变形
        y = x_enc.mul(x2_enc)
        y = y.avg_pool2d(kernel_size,stride=stride, padding=padding) * (kernel_size**2)
        x2_enc_sum = x2_enc.avg_pool2d(kernel_size, stride=stride, padding=padding) * (kernel_size**2)
        # y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
        # x2_enc.share = torch.nn.functional.avg_pool2d(x2_enc.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
        x2_enc_sum = x2_enc_sum.get_plain_text()
        x2_enc_sum = torch.where(x2_enc_sum > 1, torch.floor(x2_enc_sum), x2_enc_sum)

        z = y / x2_enc_sum 
        return z, tau
    
    def forward(self, x):
        # preprocess inputs:
        if self.stride is None:
            self.stride = self.kernel_size
        
        # perform max pooling:
        y, tau = self.approxmax_pool2d(
            x,
            kernel_size=self.kernel_size,
            padding=self.padding,
            stride=self.stride,
            dilation=self.dilation,
            ceil_mode=self.ceil_mode
        ) # 为先前写好的最大池化优化最终版
        
        # store information for backward pass and return:
        self.input = x
        self.tau = tau
        
        return y
    
#     def backward(self, grad_output):
#         # get input, kernel_size, padding, stride, dilation, ceil_mode
#         input, tau, kernel_size, padding, stride, dilation, ceil_mode = self.input, self.tau, self.kernel_size, self.padding, self.stride, self.dilation, self.ceil_mode
#         # input, tau, kernel_size, padding, stride, dilation, ceil_mode = ctx.saved_tensors
#         print(tau)
#         batches, channels, height, width = input.size()
    
#         # unfold input to adapt to calculate input_weight
#         input.share = torch.nn.functional.unfold(
#             input.share.float(), kernel_size=kernel_size, dilation=dilation, padding=padding, stride=stride
#         ) # (batch_size,channel_size*kernel_h*kernel_w,num_windows)
#         input.share = input.share.long()
#         unfolded_input = input.view(batches,channels,kernel_size**2,-1) # (batch_size,channel_size,kernel_h*kernel_w,num_windows)
    
#         # calculate input_weight
#         unfolded_input_2 = tau**2 * unfolded_input.square() + 1 # y = (ax)^2 + 1
#         unfolded_input_sum = unfolded_input_2.sum(dim=2)
#         mask_sum_max = torch.clamp(unfolded_input_sum.get_plain_text(),float('1e-2'), float('1e8')) # limit
#         input_weight = unfolded_input_2 / mask_sum_max.unsqueeze(2)
#         input_weight = input_weight.get_plain_text()
   
#         # adjust grad_output to adapt to calculate grad_input
#         grad_output = grad_output.view(batches,channels,1,-1)
    
#         # calculate grad_input
#         grad_input = grad_output*input_weight
#         grad_input = grad_input.view(batches,channels*kernel_size**2,-1)
#         grad_input.share = torch.nn.functional.fold(
#             grad_input.share.float(), (height, width), kernel_size=kernel_size, dilation=dilation, padding=padding, stride=stride
#         ) # fold grad_input to correspond input
#         grad_input.share = grad_input.share.long()
    
#         return grad_input

In [13]:
# 创建 LeNet 网络
class LeNet(crypten.nn.Sequential):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = crypten.nn.Conv2d(1, 6, kernel_size=5)
        self.relu1 = crypten.nn.ReLU()
        self.pool1 = ApproxMaxPool2D(kernel_size=2, stride=2)
        self.conv2 = crypten.nn.Conv2d(6, 16, kernel_size=5)
        self.relu2 = crypten.nn.ReLU()
        self.pool2 = ApproxMaxPool2D(kernel_size=2, stride=2)
        self.fc1 = crypten.nn.Linear(16 * 5 * 5, 120)
        self.relu3 = crypten.nn.ReLU()
        self.fc2 = crypten.nn.Linear(120, 84)
        self.relu4 = crypten.nn.ReLU()
        self.fc3 = crypten.nn.Linear(84, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = x.view(-1, 16 * 5 * 5)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        x = self.relu4(x)
        x = self.fc3(x)
        return x

In [14]:
# 创建 LeNet 模型和输入数据
model = LeNet()
model = model.encrypt()
input_data = torch.randn(32, 1, 32, 32)  # 示例输入数据，MNIST 数据集的输入形状

# 进行前向传播
encrypted_input = crypten.cryptensor(input_data)
encrypted_output = model(encrypted_input)

# 计算损失（Crypten 中的损失计算方式与 PyTorch 类似）
target = torch.randint(0, 10, (32,))
target = torch.nn.functional.one_hot(target,10)
target = crypten.cryptensor(target)
criterion = crypten.nn.CrossEntropyLoss()
loss = criterion(encrypted_output, target)

# 反向传播
loss.backward()

# 获取梯度
gradients = [param.grad for param in model.parameters()]

print(gradients)


# 进行优化等后续步骤（在 Crypten 中需要自行实现）

[MPCTensor(
	_tensor=tensor([[[[0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0]]],


        [[[0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0]]],


        [[[0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0]]],


        [[[0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0]]],


        [[[0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0]]],


        [[[0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0]]]])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
), MPCTensor(
	_tensor=tensor([0, 0, 0, 0, 0, 0])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
), MPCTens